<a href="https://colab.research.google.com/github/franklinzhou-ncsu/new_repo_554/blob/main/ST_554_Homework_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Subject: ST 554 - Homework 6

Name: Franklin Zhou

Date: 3/27/2026

# Goal

The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized or regularized models). Most homework assignments will have a part that pushes you beyond what was in the lectures! Learning to search for the right questions and browsing stackoverflow are really life skills that we should hone :) **Be sure to include markdown text describing what you are doing, even when not explicitly asked for!**

# Data

We will use a dataset from the UCI Machine Learning Repository. This data set is about wine quality (we played around with this data in class at some point I think). You can learn more about the data [here](https://archive.ics.uci.edu/ml/datasets/Wine+Quality).

The data description describes the following variables:

Input variables (based on physicochemical tests)

1 - fixed acidity

2 - volatile acidity

3 - citric acid

4 - residual sugar

5 - chlorides

6 - free sulfur dioxide

7 - total sulfur dioxide

8 - density

9 - pH

10 - sulphates

11 - alcohol

Output variable (based on sensory data):

12 - quality (score between 0 and 10)

- Rather than try to predict `quality`, let's make our target variable for fitting multiple linear regression type models `alcohol`.

- For fitting logistic regression type models we'll use the type of wine as the response variable.



# To Do:

Create a document that goes through your process of reading the data, combining it, manipulating/creating any variables, and fitting and choosing a final model for both the multiple linear regression modeling and the logistic regression modeling described below.

## Read in and Combine Data

- Read in the `winequality-red.csv` and `winequality-white.csv` files available on the [UCI machine learning repository site](https://archive.ics.uci.edu/dataset/186/wine+quality).
- Combine these two datasets and create a new variable that represents the type of wine (red or white)

In [4]:
import pandas as pd
import numpy as np

# read in the datasets
df_red_wine = pd.read_csv('winequality-red.csv', sep=';')
df_white_wine = pd.read_csv('winequality-white.csv', sep=';')

# create a new variable: red wine = 0 and white wine = 1
df_red_wine['type'] = 0
df_white_wine['type'] = 1

# combine the two datasets
df_wine = pd.concat([df_red_wine, df_white_wine])

# check the structure of the dataframe and first rows
df_wine.info()
df_wine.head()


<class 'pandas.core.frame.DataFrame'>
Index: 6497 entries, 0 to 4897
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  type                  6497 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 710.6 KB


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


## Split the Data

Split up the data set into a training and test set. For this, I want you to use stratified sampling to make sure that you have a similar proportion of white and red wines in the training and test sets. This can be done with the `train_test_split()` function ([see the help](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)).

In [5]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    df_wine.drop("alcohol", axis=1),
    df_wine["alcohol"],
    test_size=0.2,
    random_state=41,
    stratify = df_wine['type'])

## Regression Task (`alcohol` as Response)

## Train Models

- Fit four different multiple linear regression models.
    - At least one should include interaction terms
    - At least one should include some polynomial terms (you may want to standardize your predictors but that is up to you)
    - Use CV to select your best MLR model


In [88]:
from math import sqrt
#from sklearn import linear_model
#from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, Ridge, RidgeCV, ElasticNetCV, ElasticNet
from sklearn.model_selection import cross_validate

means = X_train.drop("type", axis = 1).apply(np.mean, axis = 0)
stds = X_train.drop("type", axis = 1).apply(np.std, axis = 0)

#quick function to standardize based off of a supplied mean and std
def my_std_fun (x, means, stds):
    return(x-means)/stds

# standardize the X_train data set
for x in X_train.drop("type", axis = 1).columns:
    X_train[x] = my_std_fun(X_train[x], means[x], stds[x])

# standardize the X_test data set
for x in X_test.drop("type", axis = 1).columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])


In [89]:
# Model 1 with all predictor variables
cv_full_model = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

In [90]:
# Model 2 add interaction terms
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False) # only include interaction terms
X_train_inter = poly.fit_transform(X_train)

cv_mlr_inter = cross_validate(
    LinearRegression(),
    X_train_inter,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

In [91]:
# Model 3 add polynomial terms
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly2.fit_transform(X_train)

cv_mlr_poly = cross_validate(
    LinearRegression(),
    X_train_poly,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

In [92]:
# Model 4 select some variables
X_train_select = X_train[['chlorides', 'residual sugar', 'quality', 'density']]

cv_select_model = cross_validate(
    LinearRegression(),
    X_train_select,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

In [93]:
# Comparing 4 models RMSE
print(np.sqrt(-sum(cv_full_model['test_score'])),
    np.sqrt(-sum(cv_mlr_inter['test_score'])),
    np.sqrt(-sum(cv_mlr_poly['test_score'])),
    np.sqrt(-sum(cv_select_model['test_score'])))

1.1554751418867872 0.9998743868047872 0.9523060895641579 1.8303832428265348


From the result we see that **Model 3 with polynomial terms** has the lowest RMSE, so it is the best MLR model.

In [94]:
# The best MLR model
mlr_best = LinearRegression().fit(X_train_poly, y_train)


- Fit a LASSO model with a set of predictors of your choosing
    - Use at least five predictors
    - Use CV to select the tuning parameter



In [95]:
# LASSO CV Model
lasso_mod = LassoCV(cv=5, random_state=0).fit(X_train, y_train) # Use all predictors

# check result
print(np.array(list(zip(X_train.columns, lasso_mod.coef_))))

# The best value of alpha parameter:
print(lasso_mod.alpha_)

[['fixed acidity' '0.6564732959324009']
 ['volatile acidity' '0.13619009900293194']
 ['citric acid' '0.06790715664621146']
 ['residual sugar' '1.068337051966246']
 ['chlorides' '-0.03242370699876809']
 ['free sulfur dioxide' '-0.05965364559436788']
 ['total sulfur dioxide' '-0.02492393393148446']
 ['density' '-1.9403349612023362']
 ['pH' '0.4115686468614247']
 ['sulphates' '0.1455761100939285']
 ['quality' '0.09485840210384867']
 ['type' '-1.0765149636154554']]
0.0011617160718174178


The tuning parameter $\alpha$ for **LASSO CV** is 0.00116. No variable is left out in LASSO.

In [96]:
# The best LASSO model
lasso_best = Lasso(lasso_mod.alpha_).fit(X_train,y_train)

- Fit a Ridge Regression model with a set of predictors of your choosing
    - Use at least five predictors
    - Use CV to select the tuning parameter


In [97]:
# Ridge CV Model
ridge_mod = RidgeCV(cv=5, alphas=np.logspace(-2, 2, 100))
ridge_mod.fit(X_train, y_train)

# check result
print(np.array(list(zip(X_train.columns, ridge_mod.coef_))))

# The best value of alpha parameter:
print(ridge_mod.alpha_)

[['fixed acidity' '0.6531355558837141']
 ['volatile acidity' '0.14056666700583453']
 ['citric acid' '0.07053461889388049']
 ['residual sugar' '1.061638587104564']
 ['chlorides' '-0.03545284422043825']
 ['free sulfur dioxide' '-0.059097901297360575']
 ['total sulfur dioxide' '-0.03038543749706264']
 ['density' '-1.9301132926521176']
 ['pH' '0.4108950808083856']
 ['sulphates' '0.14698156625997513']
 ['quality' '0.0974352746414766']
 ['type' '-1.054635815328501']]
6.135907273413176


The tuning parameter $\alpha$ for **Ridge CV** is 6.1359. No variable is left out in Ridge CV.

In [102]:
# The best Ridge model
ridge_best = Ridge(ridge_mod.alpha_).fit(X_train,y_train)

- Fit an Elastic Net model with a set of predictors of your choosing
    - Use at least five predictors
    - Use CV to select the tuning parameters

In [80]:
# Elastic Net CV Model
#en_mod = ElasticNetCV(cv=5, random_state=0, l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1], n_alphas = 50)
en_mod = ElasticNetCV(cv=5, random_state=0, l1_ratio = np.linspace(0.001, 0.999, 100), n_alphas = 50) # make a fine search in l1_ratio parameter
en_mod.fit(X_train, y_train)

# check result
print(np.array(list(zip(X_train.columns, en_mod.coef_))))

# The best value of alpha parameter:
print(en_mod.alpha_)
print(en_mod.l1_ratio_)

[['fixed acidity' '0.6550660986271353']
 ['volatile acidity' '0.13770177979370612']
 ['citric acid' '0.06871324578543242']
 ['residual sugar' '1.065315914505939']
 ['chlorides' '-0.033396937595935365']
 ['free sulfur dioxide' '-0.05933726539091856']
 ['total sulfur dioxide' '-0.027060961140096058']
 ['density' '-1.9359878810676718']
 ['pH' '0.41115516486730636']
 ['sulphates' '0.1460065269841559']
 ['quality' '0.09582726267244751']
 ['type' '-1.067499414821534']]
0.0012116359972490023
0.6764141414141415


The tuning parameter $\alpha$ for Elastic Net model is 0.0012, the tuning parameter L1_ratio for Elastic Net model is 0.6764.

In [101]:
# The best Elastic Net model
en_best = ElasticNet(alpha = en_mod.alpha_, l1_ratio = en_mod.l1_ratio_).fit(X_train,y_train)

## Test Models

- Using your four selected models, compare their performance on the test set.
    - Do so using RMSE as your model metric
    - Do so using MAE as your model metric

In [105]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# prepare X_test_poly for best MLR test
X_test_poly = poly2.fit_transform(X_test)

# make prediction using each model
mlr_pred = mlr_best.predict(X_test_poly)
lasso_pred = lasso_best.predict(X_test)
ridge_pred = ridge_best.predict(X_test)
en_pred = en_best.predict(X_test)


In [115]:
# create a dict for 4 models
models_pred = {
    "Best_MLR": mlr_pred,
    "LASSO": lasso_pred,
    "Ridge": ridge_pred,
    "Elastic_Net": en_pred
}

results = []

# calculate the RMSE and MAE for each model
for name, pred in models_pred.items():
    RMSE = np.sqrt(mean_squared_error(y_test, pred))
    MAE = mean_absolute_error(y_test, pred)
    results.append([name, RMSE, MAE])

# Output as a table
pd.DataFrame(results, columns=["Model", "RMSE", "MAE"])

,Model,RMSE,MAE
0,Best_MLR,0.414295,0.311780
1,LASSO,0.471538,0.356453
2,Ridge,0.471956,0.357157
3,Elastic_Net,0.471706,0.356712


From the output we can see that the Best MLR model has the best performance because it has extra polynomial terms in the model.

# Reference

[Adding Interaction Terms](https://chatgpt.com/s/t_69c7ef9e5c748191bedb3be4b5ffcacc)

[PolynomialFeatures()](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)

[Ridge CV()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)
